In [5]:
import pandas as pd
import numpy as np
import joblib
# User Input (example values)
# In a real scenario, these inputs would come from a user interface or a live data feed.
user_input = {
    'inning': 1,
    'cum_runs': 80,            # current cumulative runs
    'cum_wickets': 5,          # current wickets lost
    'overs_completed': 12.0,    # overs completed so far
    'target': 0,             # target score
    'batting_team': "Mumbai Indians",
    'bowling_team': "Chennai Super Kings",
    'venue': "Wankhede Stadium"  # assume this maps to a canonical venue (e.g., "Wankhede Stadium")
}

In [6]:
# 1. Compute Derived Features
# Compute current run rate
if user_input['overs_completed'] > 0:
    current_run_rate = user_input['cum_runs'] / user_input['overs_completed']
else:
    current_run_rate = 0

# Compute required run rate
remaining_overs = 20 - user_input['overs_completed']
if user_input['inning'] == 2 and remaining_overs > 0:
    required_run_rate = (user_input['target'] - user_input['cum_runs']) / remaining_overs
else:
    required_run_rate = 0

In [8]:
# 2. Load Saved Encoders and Model
# Load the team and venue LabelEncoders (ensure these files were saved during training)
le_team = joblib.load('le_team.pkl')
le_venue = joblib.load('le_venue.pkl')

# Load the final trained model (e.g., a Random Forest model)
model = joblib.load('final_rf_model.pkl')


In [9]:
# 3. Encode Categorical Variables
# Encode batting team and bowling team using the team encoder
batting_team_encoded = le_team.transform([user_input['batting_team']])[0]
bowling_team_encoded = le_team.transform([user_input['bowling_team']])[0]

# Encode the venue using the venue encoder
venue_canonical_encoded = le_venue.transform([user_input['venue']])[0]

# 4. Create Input DataFrame for Inference
# The final features (order matters) are:
# ['inning', 'cum_runs', 'cum_wickets', 'current_run_rate',
#  'required_run_rate', 'target', 'batting_team_encoded',
#  'bowling_team_encoded', 'venue_canonical_encoded']
input_data = {
    'inning': [user_input['inning']],
    'cum_runs': [user_input['cum_runs']],
    'cum_wickets': [user_input['cum_wickets']],
    'current_run_rate': [current_run_rate],
    'required_run_rate': [required_run_rate],
    'target': [user_input['target']],
    'batting_team_encoded': [batting_team_encoded],
    'bowling_team_encoded': [bowling_team_encoded],
    'venue_canonical_encoded': [venue_canonical_encoded]
}

input_df = pd.DataFrame(input_data)

In [16]:
# 5. Make Prediction
prediction = model.predict(input_df)[0]
predicted_probabilities = model.predict_proba(input_df)[0]

# 6. Map Prediction to Team Names with Probability Ranges
batting_team = user_input['batting_team']
bowling_team = user_input['bowling_team']

win_prob = predicted_probabilities[1] * 100  # Probability of batting team winning
loss_prob = predicted_probabilities[0] * 100  # Probability of bowling team winning

if prediction == 1:
    predicted_winner = batting_team
    predicted_loser = bowling_team
else:
    predicted_winner = bowling_team
    predicted_loser = batting_team
    win_prob, loss_prob = loss_prob, win_prob  # Swap probabilities to match winner/loser

# Print results in a clear format
print(f"🏆 Predicted Winner: {predicted_winner}")
print(f"❌ Predicted Loser: {predicted_loser}")
print(f"🔢 Probability Range To Win: {predicted_winner} ({win_prob:.2f}%) | {predicted_loser} ({loss_prob:.2f}%)")


🏆 Predicted Winner: Chennai Super Kings
❌ Predicted Loser: Mumbai Indians
🔢 Probability Range To Win: Chennai Super Kings (99.50%) | Mumbai Indians (0.50%)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
